<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D1%81%D1%8B%D1%80%D1%8B%D1%85__%D0%A7%D0%B0%D1%81%D1%82%D1%8C_1_ipynb%22%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Цель работы:**  
Создание модели, позволяющей спрогнозировать состав следующего заказа пользователя на основе его истории покупок. Модель улучшит персонализацию сервиса и повысит качество обслуживания клиентов.


# Постановка задачи
Необходимо разработать систему рекомендаций, которая будет определять вероятность включения определенной категории товаров в будущий заказ пользователя. Задача решается как многоклассовый классификатор с использованием метрики F1-score.

# Описание набора данных

В проекте используется история заказов 20 000 пользователей, разделённая на тренировочную и тестовую выборки по дате. Тестовая выборка содержит заказы после определённой даты отсечки.

Основной тренировочный файл содержит следующие данные:  
- **user_id** — уникальный идентификатор пользователя  
- **order_completed_at** — дата и время завершения заказа  
- **cart** — категория товара, входящего в заказ (уникальные категории)

Задача — для каждой пары (пользователь, категория), встречающейся в тестовой выборке, предсказать бинарный признак: будет ли категория присутствовать в следующем заказе пользователя.

Идентификаторы пар представлены в формате `"{user_id};{category_id}"`, что учитывается при обработке данных.

Данные подготовлены на основе истории заказов с учётом особенностей временного разделения выборок.


In [21]:
# Подключение Google Drive (оставлено без изменений)
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
from collections import Counter

# Пути и структура проекта
project_path = "/content/drive/MyDrive/Colab Notebooks/sm"
folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "src/models",
    "src/utils",
    "scripts",
]
for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

# Функция для вывода структуры папок
def print_tree(root, prefix=""):
    files = sorted(os.listdir(root))
    for i, name in enumerate(files):
        path = os.path.join(root, name)
        is_last = (i == len(files) - 1)
        branch = "└── " if is_last else "├── "
        print(prefix + branch + name + ("/" if os.path.isdir(path) else ""))
        if os.path.isdir(path):
            print_tree(path, prefix + ("    " if is_last else "│   "))

print("\n=== Структура проекта (project_path) ===")
print_tree(project_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Структура проекта (project_path) ===
├── data/
│   ├── processed/
│   │   └── full_orders.parquet
│   └── raw/
│       ├── sample_submission.csv
│       └── train.csv
├── notebooks/
├── plan/
├── scripts/
└── src/
    ├── models/
    └── utils/


In [22]:
# Загрузка данных
raw_data_path = os.path.join(project_path, "data/raw")
train = pd.read_csv(os.path.join(raw_data_path, "train.csv"))
sub = pd.read_csv(os.path.join(raw_data_path, "sample_submission.csv"))

# Информация о данных
print("Информация о train.csv:")
print(train.info())
print(f"Уникальных пользователей: {train['user_id'].nunique()}")
print(f"Уникальных категорий: {train['cart'].nunique()}")

print(train.head())

Информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None
Уникальных пользователей: 20000
Уникальных категорий: 881
   user_id   order_completed_at  cart
0        2  2015-03-22 09:25:46   399
1        2  2015-03-22 09:25:46    14
2        2  2015-03-22 09:25:46   198
3        2  2015-03-22 09:25:46    88
4        2  2015-03-22 09:25:46   157


In [24]:
# Преобразование даты один раз
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print(f"Период данных: с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")

Период данных: с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [25]:
print("\nИнформация о sample_submission.csv:")
print(sub.info())


Информация о sample_submission.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [26]:
# Топ-30 категорий за весь период
cat_counter = Counter(train['cart'])
top30 = cat_counter.most_common(30)
top30_cats = [c for c, _ in top30]
print("\nТоп-30 категорий по всем пользователям:", top30_cats)


Топ-30 категорий по всем пользователям: [57, 14, 61, 398, 23, 84, 22, 409, 17, 402, 55, 383, 420, 382, 19, 430, 41, 169, 425, 16, 9, 88, 384, 42, 395, 5, 432, 54, 89, 29]


In [27]:
# Статистика заказов на пользователя
orders_per_user = train['user_id'].value_counts()
print("\nДополнительная статистика по train.csv:")
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())


Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64


In [28]:
# Анализ временных периодов (помесячно)
print("\n=== Анализ временных периодов (помесячно) ===\n")
train['year_month'] = train['order_completed_at'].dt.to_period('M')
monthly_stats = train.groupby('year_month').agg(
    total_orders=('order_completed_at', 'count'),
    first_order=('order_completed_at', 'min'),
    last_order=('order_completed_at', 'max'),
    unique_users=('user_id', 'nunique')
)
actual_days = train.groupby('year_month')['order_completed_at'].apply(lambda x: x.dt.date.nunique())
days_in_month = monthly_stats['first_order'].dt.days_in_month
monthly_stats['missing_days'] = days_in_month - actual_days
monthly_stats['stability'] = monthly_stats['missing_days'].apply(
    lambda x: 'стабилен' if x == 0 else f'нестабилен ({int(x)} пропусков)'
)

for idx, row in monthly_stats.iterrows():
    print(f"Период {idx.strftime('%Y-%m')}: {row['first_order']} по {row['last_order']} - {row['stability']} - {int(row['total_orders']):,} заказов")

print(f"\nВсего периодов: {len(monthly_stats)}")
print(f"Общее количество заказов: {monthly_stats['total_orders'].sum():,}")
print(f"Среднее количество заказов в месяц: {monthly_stats['total_orders'].mean():.0f}")


=== Анализ временных периодов (помесячно) ===

Период 2015-03: 2015-03-22 09:25:46 по 2015-03-22 09:25:46 - нестабилен (30 пропусков) - 16 заказов
Период 2015-06: 2015-06-18 16:15:33 по 2015-06-18 16:15:33 - нестабилен (29 пропусков) - 1 заказов
Период 2015-07: 2015-07-04 14:05:22 по 2015-07-22 21:26:56 - нестабилен (28 пропусков) - 17 заказов
Период 2015-08: 2015-08-12 10:33:44 по 2015-08-12 10:33:44 - нестабилен (30 пропусков) - 2 заказов
Период 2015-11: 2015-11-27 19:37:17 по 2015-11-27 19:37:17 - нестабилен (29 пропусков) - 1 заказов
Период 2015-12: 2015-12-01 14:30:59 по 2015-12-14 10:30:14 - нестабилен (29 пропусков) - 15 заказов
Период 2016-04: 2016-04-03 12:03:59 по 2016-04-24 15:59:48 - нестабилен (27 пропусков) - 21 заказов
Период 2016-05: 2016-05-11 13:38:25 по 2016-05-27 19:21:46 - нестабилен (28 пропусков) - 36 заказов
Период 2016-06: 2016-06-04 14:25:03 по 2016-06-07 13:51:38 - нестабилен (28 пропусков) - 10 заказов
Период 2016-07: 2016-07-01 14:51:10 по 2016-07-30 13:11

In [31]:
# Сравнение топ-30 категорий за весь период и август 2020
print("\n=== Сравнение топ-30 категорий за весь период и за август 2020 ===\n")
print("Топ-30 категорий за ВЕСЬ период:", top30_cats)

august_2020_data = train[train['order_completed_at'].dt.to_period('M') == '2020-08']
august_cat_counter = Counter(august_2020_data['cart'])
august_top30 = august_cat_counter.most_common(30)
august_top30_cats = [c for c, _ in august_top30]
print("Топ-30 категорий за АВГУСТ 2020:", august_top30_cats)

common_categories = set(top30_cats) & set(august_top30_cats)
print(f"\nПересечение топ-30: {len(common_categories)} общих категорий")
print("Общие категории:", sorted(common_categories))

unique_to_full = set(top30_cats) - set(august_top30_cats)
print(f"\nУникальные для ВСЕГО периода: {len(unique_to_full)} категорий")
print("Уникальные категории:", sorted(unique_to_full))

unique_to_august = set(august_top30_cats) - set(top30_cats)
print(f"\nУникальные для АВГУСТА 2020: {len(unique_to_august)} категорий")
print("Уникальные категории:", sorted(unique_to_august))


=== Сравнение топ-30 категорий за весь период и за август 2020 ===

Топ-30 категорий за ВЕСЬ период: [57, 14, 61, 398, 23, 84, 22, 409, 17, 402, 55, 383, 420, 382, 19, 430, 41, 169, 425, 16, 9, 88, 384, 42, 395, 5, 432, 54, 89, 29]
Топ-30 категорий за АВГУСТ 2020: [57, 14, 61, 398, 23, 84, 22, 17, 409, 55, 402, 430, 382, 383, 19, 420, 41, 16, 425, 169, 9, 88, 42, 432, 384, 395, 5, 54, 82, 388]

Пересечение топ-30: 28 общих категорий
Общие категории: [5, 9, 14, 16, 17, 19, 22, 23, 41, 42, 54, 55, 57, 61, 84, 88, 169, 382, 383, 384, 395, 398, 402, 409, 420, 425, 430, 432]

Уникальные для ВСЕГО периода: 2 категорий
Уникальные категории: [29, 89]

Уникальные для АВГУСТА 2020: 2 категорий
Уникальные категории: [82, 388]


In [32]:
print("\n=== Анализ позиций в рейтинге ===\n")
print("Категория | Ранг(весь период) | Ранг(август 2020) | Количество(весь период)")
print("-" * 70)
for i, (cat, count) in enumerate(top30, 1):
    august_rank = next((j for j, (ac, _) in enumerate(august_top30, 1) if ac == cat), None)
    aug_rank_str = str(august_rank) if august_rank else "нет в топ-30"
    print(f"{cat:9} | {i:17} | {aug_rank_str:16} | {count:>24,}")


=== Анализ позиций в рейтинге ===

Категория | Ранг(весь период) | Ранг(август 2020) | Количество(весь период)
----------------------------------------------------------------------
       57 |                 1 | 1                |                  108,877
       14 |                 2 | 2                |                   93,957
       61 |                 3 | 3                |                   91,543
      398 |                 4 | 4                |                   81,694
       23 |                 5 | 5                |                   71,837
       84 |                 6 | 6                |                   68,715
       22 |                 7 | 7                |                   68,478
      409 |                 8 | 9                |                   59,920
       17 |                 9 | 8                |                   58,840
      402 |                10 | 11               |                   49,925
       55 |                11 | 10               |       

In [37]:
print("\n=== Анализ продаж категорий 89 и 29 за последний год (с августа 2019 по август 2020) ===\n")
last_year_data = train[(train['order_completed_at'] >= '2019-08-01') & (train['order_completed_at'] <= '2020-08-31')].copy()
last_year_data['month'] = last_year_data['order_completed_at'].dt.to_period('M')
monthly_sales_29_89 = last_year_data[last_year_data['cart'].isin([29, 89])].groupby(['month', 'cart']).size().unstack(fill_value=0)

print("Месяц      | Кат.29 | Кат.89")
print("----------------------------")
for month in monthly_sales_29_89.index:
    print(f"{month} | {monthly_sales_29_89.loc[month, 29]} | {monthly_sales_29_89.loc[month, 89]}")

print(f"\n=== Анализ категорий 82 и 388 (уникальные для августа) ===\n")
monthly_sales_82_388 = last_year_data[last_year_data['cart'].isin([82, 388])].groupby(['month', 'cart']).size().unstack(fill_value=0)

print("Месяц      | Кат.82 | Кат.388")
print("-----------------------------")
for month in monthly_sales_82_388.index:
    cat82 = monthly_sales_82_388.loc[month, 82] if 82 in monthly_sales_82_388.columns else 0
    cat388 = monthly_sales_82_388.loc[month, 388] if 388 in monthly_sales_82_388.columns else 0
    print(f"{month} | {cat82} | {cat388}")



=== Анализ продаж категорий 89 и 29 за последний год (с августа 2019 по август 2020) ===

Месяц      | Кат.29 | Кат.89
----------------------------
2019-08 | 265 | 314
2019-09 | 454 | 547
2019-10 | 1083 | 1175
2019-11 | 1457 | 1537
2019-12 | 1319 | 1420
2020-01 | 1228 | 1298
2020-02 | 1192 | 1367
2020-03 | 1646 | 1977
2020-04 | 2290 | 2457
2020-05 | 2981 | 3173
2020-06 | 4304 | 3710
2020-07 | 4199 | 3764
2020-08 | 3755 | 3771

=== Анализ категорий 82 и 388 (уникальные для августа) ===

Месяц      | Кат.82 | Кат.388
-----------------------------
2019-08 | 282 | 232
2019-09 | 484 | 429
2019-10 | 1150 | 901
2019-11 | 1570 | 1165
2019-12 | 1261 | 1013
2020-01 | 1137 | 987
2020-02 | 1211 | 1104
2020-03 | 1661 | 1454
2020-04 | 2165 | 2011
2020-05 | 3014 | 2820
2020-06 | 3717 | 6039
2020-07 | 4047 | 3909
2020-08 | 3899 | 3908


In [35]:
# Функция для определения активных и стабильных пользователей и топ-30 категорий
def stable_users_and_top_categories(df, start_date, end_date, period_name):
    filtered = df[(df['order_completed_at'] >= start_date) & (df['order_completed_at'] <= end_date)].copy()
    filtered.set_index('order_completed_at', inplace=True)
    active_users = (
        filtered.groupby('user_id')
        .resample('MS')
        .size()
        .reset_index(name='counts')
    )
    pivot = active_users.pivot(index='user_id', columns='order_completed_at', values='counts')
    active_list = pivot.dropna(axis=0, how='any').index.tolist()
    print(f"Количество активных пользователей за {period_name}: {len(active_list)}")

    std_results = {}
    for user_id in active_list:
        user_data = filtered[filtered['user_id'] == user_id]
        basket_sizes = user_data.groupby('order_completed_at')['cart'].nunique()
        std_results[user_id] = basket_sizes.std()

    std_df = pd.DataFrame(list(std_results.items()), columns=['user_id', 'Basket_Size_STD'])
    median_std = std_df['Basket_Size_STD'].median()
    stable = std_df[std_df['Basket_Size_STD'] <= median_std]['user_id']
    print(f"Медиана стандартного отклонения размера корзины за {period_name}: {median_std:.4f}")
    print(f"Количество стабильных пользователей за {period_name}: {len(stable)}")
    print(f"Примеры стабильных пользователей и их стандартные отклонения:")
    print(std_df[std_df['user_id'].isin(stable)].head())

    stable_data = filtered[filtered['user_id'].isin(stable)]
    top30_counts = stable_data['cart'].value_counts().head(30)
    top30_list = top30_counts.index.tolist()
    print(f"ТОП-30 категорий для стабильных пользователей за {period_name}: {top30_list}")
    return top30_list

# Анализ для 12, 6 и 3 месяцев
top_30_cats_12m = stable_users_and_top_categories(train, '2019-10-01', '2020-08-31', '12 месяцев')
top_30_cats_6m = stable_users_and_top_categories(train, '2020-03-01', '2020-08-31', '6 месяцев')
top_30_cats_3m = stable_users_and_top_categories(train, '2020-06-01', '2020-08-31', '3 месяца')


Количество активных пользователей за 12 месяцев: 1871
Медиана стандартного отклонения размера корзины за 12 месяцев: 5.8930
Количество стабильных пользователей за 12 месяцев: 936
Примеры стабильных пользователей и их стандартные отклонения:
    user_id  Basket_Size_STD
5        38         2.528654
9        59         3.132016
15       75         3.872439
19       85         2.187885
21       91         3.576930
ТОП-30 категорий для стабильных пользователей за 12 месяцев: [57, 61, 14, 398, 23, 22, 84, 409, 55, 17, 402, 420, 430, 16, 41, 19, 382, 383, 169, 425, 9, 88, 384, 5, 42, 395, 89, 432, 54, 82]
Количество активных пользователей за 6 месяцев: 3880
Медиана стандартного отклонения размера корзины за 6 месяцев: 5.2071
Количество стабильных пользователей за 6 месяцев: 1940
Примеры стабильных пользователей и их стандартные отклонения:
    user_id  Basket_Size_STD
1        12         4.195235
2        16         5.024080
6        30         4.509250
7        38         2.887764
15       

In [36]:
# Сравнение топ-30 категорий с общим ТОП-30
print(f"ТОП-30 категорий за ВЕСЬ период: {top30_cats}")

def compare_top30(period_top30, period_name):
    intersect = set(period_top30) & set(top30_cats)
    unique_to_period = set(period_top30) - intersect
    unique_to_total = set(top30_cats) - intersect
    print(f"Перекрытие с общим ТОП-30 за {period_name}: {len(intersect)} категорий")
    print(f"Уникальные категории за {period_name}: {list(unique_to_period)}")
    print(f"Категории общего ТОП-30 отсутствующие в {period_name}: {list(unique_to_total)}")

compare_top30(top_30_cats_12m, '12 месяцев')
compare_top30(top_30_cats_6m, '6 месяцев')
compare_top30(top_30_cats_3m, '3 месяца')

ТОП-30 категорий за ВЕСЬ период: [57, 14, 61, 398, 23, 84, 22, 409, 17, 402, 55, 383, 420, 382, 19, 430, 41, 169, 425, 16, 9, 88, 384, 42, 395, 5, 432, 54, 89, 29]
Перекрытие с общим ТОП-30 за 12 месяцев: 29 категорий
Уникальные категории за 12 месяцев: [82]
Категории общего ТОП-30 отсутствующие в 12 месяцев: [29]
Перекрытие с общим ТОП-30 за 6 месяцев: 29 категорий
Уникальные категории за 6 месяцев: [100]
Категории общего ТОП-30 отсутствующие в 6 месяцев: [29]
Перекрытие с общим ТОП-30 за 3 месяца: 28 категорий
Уникальные категории за 3 месяца: [396, 388]
Категории общего ТОП-30 отсутствующие в 3 месяца: [89, 29]
